In [1]:
import os
import json
import csv
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_fscore_support

import torch as th
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

METRIC_KEYS = ("AP", "P", "R", "F1", "PA_F1", "EP", "ER", "EF1", "FP1000")

# Authors' hyperparameters (kanad/config.toml, [Model_Params.Default])
AUTHOR_PARAMS = dict(window=96, order=2, batch_size=1024, epochs=100, lr=0.01)
EARLY_STOP_PATIENCE = 3          # authors: EarlyStoppingTorch(patience=3)
EARLY_STOP_DELTA = 1e-4          # authors: EarlyStoppingTorch default delta
VALID_FRACTION_OF_TRAIN = 0.2    # authors: valid_proportion = 0.2

In [2]:
# make it interpretable
class KANADModel(nn.Module):
    def __init__(self, window: int, order: int, *args, **kwargs) -> None:
        super().__init__()
        self.window = window
        self.order = order
        self.channels = 2 * self.order + 1
        self.register_buffer(
            "orders",
            self._create_custom_periodic_cosine().unsqueeze(0),  # (1, order, window)
        )
        self.out_conv = nn.Conv1d(self.channels, 1, 1, bias=False)
        self.act = nn.GELU()
        self.bn1 = nn.BatchNorm1d(self.channels)
        self.bn3 = nn.BatchNorm1d(1)
        self.bn2 = nn.BatchNorm1d(self.channels)
        self.init_conv = nn.Conv1d(self.channels, self.channels, 3, 1, 1, bias=False)
        self.inner_conv = nn.Conv1d(self.channels, self.channels, 3, 1, 1, bias=False)
        self.final_conv = nn.Conv1d(1, 1, window, padding=0, stride=1, dilation=1)

    def forward(
        self, x: th.Tensor, *args, **kwargs
    ) -> th.Tensor | tuple[th.Tensor, th.Tensor]:
        res = []
        res.append(x.unsqueeze(1))
        ff = th.concat(
            [self.orders.repeat(x.size(0), 1, 1)]  # type: ignore
            + [th.cos(order * x.unsqueeze(1)) for order in range(1, self.order + 1)]
            + [x.unsqueeze(1)],
            dim=1,
        )  # batch,self.channel,window
        res.append(ff)
        ff = self.init_conv(ff)
        ff = self.bn1(ff)
        ff = self.act(ff)
        ff = self.inner_conv(ff) + res.pop()
        ff = self.bn2(ff)
        ff = self.act(ff)
        ff = self.out_conv(ff) + res.pop()
        ff = self.bn3(ff)
        ff = self.act(ff)
        ff = self.final_conv(ff)
        return ff.squeeze(1)

    def _create_custom_periodic_cosine(self) -> th.Tensor:
        pl = [i for i in range(1, self.order + 1)]
        result = th.empty(self.order, self.window, dtype=th.float32)
        for i, p in enumerate(pl):
            range_value = th.arange(self.window, dtype=th.float32)
            result[i, :] = th.cos(2 * th.pi * range_value * p / self.window)
        return result

In [3]:
# Numenta Anomaly Benchmark data loading


@dataclass
class Series:
    name: str
    values: np.ndarray
    labels: np.ndarray


def _labels_from_windows(timestamps, windows):
    labels = np.zeros(len(timestamps), dtype=np.int64)
    if not windows:
        return labels
    ts = pd.to_datetime(timestamps)
    for start, end in windows:
        labels[(ts >= pd.to_datetime(start)) & (ts <= pd.to_datetime(end))] = 1
    return labels


def load_all_series(nab_root):
    labels_path = os.path.join(nab_root, "labels", "combined_windows.json")
    data_dir = os.path.join(nab_root, "data")
    with open(labels_path) as f:
        windows_map = json.load(f)
    series = []
    for rel_name, windows in sorted(windows_map.items()):
        csv_path = os.path.join(data_dir, rel_name)
        if not os.path.isfile(csv_path):
            csv_path = os.path.join(data_dir, *rel_name.split("/"))
        if not os.path.isfile(csv_path):
            print(f"[warn] missing CSV for {rel_name}, skipping"); continue
        df = pd.read_csv(csv_path)
        series.append(Series(rel_name, df["value"].to_numpy(dtype=np.float64),
                             _labels_from_windows(df["timestamp"], windows)))
    if not series:
        raise FileNotFoundError(f"No NAB series loaded from {nab_root}.")
    print(f"[data] loaded {len(series)} NAB series from {nab_root}")
    return series

In [4]:
#split, preprocessing and windows


def fits_split(length):
    """30% train / 20% validation / 50% test with floor boundaries."""
    train_end = int(length * 0.30)
    val_end = int(length * 0.50)
    if train_end < 2 or train_end >= val_end or val_end >= length:
        raise ValueError(f"Series too short ({length}) for the requested split.")
    return train_end, val_end


def preprocess_authors(values, train_end):
    """Authors' preprocessing: z-score (training-partition statistics), then
    first-order differencing with edge padding (diff_order = 1)."""
    x = np.asarray(values, dtype=np.float64)
    mu = x[:train_end].mean()
    sd = x[:train_end].std()
    sd = sd if sd >= 1e-8 else 1.0
    z = (x - mu) / sd
    d = np.pad(z, (0, 1), "edge") - np.pad(z, (1, 0), "edge")
    return d[:-1]


def next_step_windows(d, lo, hi, T, allow_context):
    """(window of T past values, next value) for target positions in [lo, hi).
    allow_context=False: the whole window stays inside [lo, hi) (training).
    allow_context=True : the window may reach back before lo (validation, test)."""
    first = max(lo if allow_context else lo + T, T)
    if first >= hi:
        return np.empty((0, T)), np.empty((0,)), np.empty((0,), dtype=int)
    idx = np.arange(first, hi)
    view = np.lib.stride_tricks.sliding_window_view(d, T)
    X = np.ascontiguousarray(view[idx - T])
    return X, d[idx], idx

In [5]:
#training (the authors loop) and prediction


class EarlyStopping:
    """Same logic as the authors' EarlyStoppingTorch with save_path=None
    (no checkpoint restore; training simply stops)."""
    def __init__(self, patience=EARLY_STOP_PATIENCE, delta=EARLY_STOP_DELTA):
        self.patience, self.delta = patience, delta
        self.counter, self.best_score, self.early_stop = 0, None, False

    def __call__(self, val_loss):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0


def fit_kanad(d, train_end, params, device):
    T = params["window"]
    split_idx = int(train_end * VALID_FRACTION_OF_TRAIN)
    n_sub = train_end - split_idx
    Xtr, ytr, _ = next_step_windows(d, 0, n_sub, T, allow_context=False)
    Xva, yva, _ = next_step_windows(d, n_sub, train_end, T, allow_context=False)
    if Xtr.shape[0] == 0:
        raise ValueError(f"no training windows (training part too short for T={T}).")

    model = KANADModel(window=T, order=params["order"]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=params["lr"])
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.75)
    loss_fn = nn.MSELoss()
    stopper = EarlyStopping()

    xt = th.tensor(Xtr, dtype=th.float32); yt = th.tensor(ytr, dtype=th.float32).unsqueeze(1)
    train_loader = DataLoader(TensorDataset(xt, yt), batch_size=params["batch_size"], shuffle=True)
    use_valid = Xva.shape[0] > 0
    if use_valid:
        xv = th.tensor(Xva, dtype=th.float32); yv = th.tensor(yva, dtype=th.float32).unsqueeze(1)
        valid_loader = DataLoader(TensorDataset(xv, yv), batch_size=params["batch_size"], shuffle=False)

    for epoch in range(1, params["epochs"] + 1):
        model.train(mode=True)
        avg_train = 0.0
        for x, target in train_loader:
            x, target = x.to(device), target.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x), target)
            loss.backward()
            optimizer.step()
            avg_train += loss.cpu().item()
        avg_train /= max(len(train_loader), 1)

        if use_valid:
            model.eval()
            avg_valid = 0.0
            with th.no_grad():
                for x, target in valid_loader:
                    x, target = x.to(device), target.to(device)
                    avg_valid += loss_fn(model(x), target).cpu().item()
            monitored = avg_valid / max(len(valid_loader), 1)
        else:
            # Not covered by the authors' code: with too few points there are no
            # validation windows, so the training loss is monitored instead.
            monitored = avg_train
        scheduler.step()
        stopper(monitored)
        if stopper.early_stop:
            break
    return model, use_valid


def predict_kanad(model, X, device, batch_size):
    model.eval()
    out = []
    with th.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = th.tensor(X[i:i + batch_size], dtype=th.float32).to(device)
            out.append(model(xb).cpu().numpy().reshape(-1))
    return np.concatenate(out) if out else np.empty(0)

In [6]:
# scoring one series


@dataclass
class SeriesScoring:
    name: str
    val_scores: np.ndarray
    val_labels: np.ndarray
    test_scores: np.ndarray
    test_labels: np.ndarray
    used_valid_windows: bool


def _scores_for_block(d, labels, lo, hi, T, model, device, batch_size):
    X, y, idx = next_step_windows(d, lo, hi, T, allow_context=True)
    if X.shape[0] == 0:
        return np.empty(0), np.empty(0, dtype=np.int64)
    pred = predict_kanad(model, X, device, batch_size)
    scores = np.abs(pred - y)                 # authors: |output - target|
    scores[np.isnan(scores)] = 1000           # authors: NaN scores -> 1000
    return scores, labels[idx].astype(np.int64)


def score_series(name, values, labels, params, device):
    values = np.asarray(values, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    train_end, val_end = fits_split(values.shape[0])
    d = preprocess_authors(values, train_end)
    model, used_valid = fit_kanad(d, train_end, params, device)
    T, bs = params["window"], params["batch_size"]
    vs, vl = _scores_for_block(d, labels, train_end, val_end, T, model, device, bs)
    ts, tl = _scores_for_block(d, labels, val_end, values.shape[0], T, model, device, bs)
    return SeriesScoring(name, vs, vl, ts, tl, used_valid)

In [7]:
#metrics (including event-level and false alarms)


from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching


def percentile_threshold(scores, q=99.0):
    """Label-free threshold: the q-th percentile of the validation scores."""
    scores = np.asarray(scores, dtype=np.float64)
    return float(np.percentile(scores, q)) if scores.size else np.inf


def point_adjust(pred, labels):
    pred = np.asarray(pred, dtype=np.int64).copy()
    labels = np.asarray(labels, dtype=np.int64)
    n = labels.shape[0]
    i = 0
    while i < n:
        if labels[i] == 1:
            j = i
            while j < n and labels[j] == 1:
                j += 1
            if pred[i:j].any():
                pred[i:j] = 1
            i = j
        else:
            i += 1
    return pred


def intervals(binary):
    """Contiguous runs of 1s as (start, end_exclusive) pairs."""
    b = np.asarray(binary, dtype=np.int8)
    if b.size == 0:
        return []
    d = np.diff(np.concatenate(([0], b, [0])))
    return list(zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1)))


def event_scores(pred, labels):
    """Event-level precision, recall and F1 using maximum one-to-one matching
    between labelled intervals and alarm intervals. A labelled interval and an
    alarm interval can be matched when they share at least one point; each
    interval is used in at most one match."""
    true_iv, alarm_iv = intervals(labels), intervals(pred)
    n_true, n_alarm = len(true_iv), len(alarm_iv)
    matched = 0
    if n_true and n_alarm:
        rows, cols = [], []
        for i, (ts, te) in enumerate(true_iv):
            for j, (a_s, a_e) in enumerate(alarm_iv):
                if a_s < te and ts < a_e:
                    rows.append(i); cols.append(j)
        if rows:
            graph = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(n_true, n_alarm))
            matching = maximum_bipartite_matching(graph, perm_type="column")
            matched = int((matching >= 0).sum())
    ep = matched / n_alarm if n_alarm else 0.0
    er = matched / n_true if n_true else 0.0
    ef1 = 2 * ep * er / (ep + er) if (ep + er) > 0 else 0.0
    return ep, er, ef1


@dataclass
class SeriesResult:
    name: str
    ap: float = np.nan
    precision: float = np.nan
    recall: float = np.nan
    f1: float = np.nan
    pa_f1: float = np.nan
    ep: float = np.nan          # event precision
    er: float = np.nan          # event recall
    ef1: float = np.nan         # event F1
    fp1000: float = np.nan      # flagged points per 1,000 observations (series without positives)
    n_test_points: int = 0
    n_test_anom: int = 0
    valid_ap: bool = False
    valid_prf: bool = False
    valid_fp: bool = False


def evaluate_series(name, val_scores, test_scores, test_labels):
    res = SeriesResult(name=name)
    test_labels = np.asarray(test_labels, dtype=np.int64)
    res.n_test_points = int(test_labels.size)
    res.n_test_anom = int(test_labels.sum())
    if res.n_test_anom > 0 and test_scores.size > 0:
        res.ap = float(average_precision_score(test_labels, test_scores))
        res.valid_ap = True
    threshold = percentile_threshold(val_scores, 99.0)
    if test_scores.size > 0:
        # A point is flagged when its score EXCEEDS the threshold (strictly greater).
        pred = (np.asarray(test_scores) > threshold).astype(np.int64)
        if res.n_test_anom > 0:
            p, r, f1, _ = precision_recall_fscore_support(
                test_labels, pred, average="binary", zero_division=0)
            res.precision, res.recall, res.f1 = float(p), float(r), float(f1)
            pa_pred = point_adjust(pred, test_labels)
            _, _, pa_f1, _ = precision_recall_fscore_support(
                test_labels, pa_pred, average="binary", zero_division=0)
            res.pa_f1 = float(pa_f1)
            res.ep, res.er, res.ef1 = event_scores(pred, test_labels)
            res.valid_prf = True
        else:
            # No labelled anomaly in the test part: every flagged point is a false alarm.
            res.fp1000 = 1000.0 * float(pred.sum()) / pred.size
            res.valid_fp = True
    return res

def macro_average(results):
    ap = [r.ap for r in results if r.valid_ap]
    prf = [r for r in results if r.valid_prf]
    fp = [r.fp1000 for r in results if r.valid_fp]
    mean = lambda v: float(np.mean(v)) if len(v) else np.nan
    return {
        "AP": mean(ap),
        "P": mean([r.precision for r in prf]),
        "R": mean([r.recall for r in prf]),
        "F1": mean([r.f1 for r in prf]),
        "PA_F1": mean([r.pa_f1 for r in prf]),
        "EP": mean([r.ep for r in prf]),
        "ER": mean([r.er for r in prf]),
        "EF1": mean([r.ef1 for r in prf]),
        "FP1000": mean(fp),
        "n_series_total": len(results),
        "n_series_ap": len(ap),
        "n_series_prf": len(prf),
        "n_series_no_pos": len(fp),
        "n_test_points_total": int(sum(r.n_test_points for r in results)),
    }


def summarise_across_seeds(per_seed_macro):
    out = {}
    for key in METRIC_KEYS:
        vals = np.array([m[key] for m in per_seed_macro], dtype=np.float64)
        out[key + "_mean"] = float(np.nanmean(vals))
        out[key + "_std"] = float(np.nanstd(vals, ddof=1))  # sample standard deviation
    return out

In [8]:
#runner and result files


def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); th.manual_seed(seed)
    if th.cuda.is_available():
        th.cuda.manual_seed_all(seed)


def run_nab(nab_root, out="results_kanad_authors", params=None,
            seeds=(42, 43, 44), device=None, max_series=None):
    params = dict(AUTHOR_PARAMS if params is None else params)
    os.makedirs(out, exist_ok=True)
    device = device or ("cuda" if th.cuda.is_available() else "cpu")
    n_params = sum(p.numel() for p in KANADModel(params["window"], params["order"]).parameters())
    print(f"[device] {device}   [params] {params}   [trainable parameters] {n_params}")
    raw = [(s.name, s.values, s.labels) for s in load_all_series(nab_root)]
    if max_series:
        raw = raw[:max_series]
        print(f"[note] limited to first {len(raw)} series for a quick run")

    per_seed_macro, per_series_rows = [], []
    for seed in seeds:
        set_all_seeds(seed)
        results, n_no_valid = [], 0
        for i, (name, values, labels) in enumerate(raw, 1):
            try:
                sc = score_series(name, values, labels, params, device)
            except ValueError as e:
                print(f"[skip] seed {seed} {name}: {e}"); continue
            n_no_valid += int(not sc.used_valid_windows)
            res = evaluate_series(name, sc.val_scores, sc.test_scores, sc.test_labels)
            results.append(res)
            per_series_rows.append((seed, res.name, res.ap, res.precision, res.recall,
                                    res.f1, res.pa_f1, res.ep, res.er, res.ef1, res.fp1000,
                                    res.n_test_points, res.n_test_anom))
            if res.valid_ap:
                print(f"  seed {seed} [{i}/{len(raw)}] {name}: "
                      f"AP={res.ap:.3f} F1={res.f1:.3f} EF1={res.ef1:.3f}")
            else:
                print(f"  seed {seed} [{i}/{len(raw)}] {name}: (no test anomaly) "
                      f"false alarm points per 1,000 = {res.fp1000:.2f}")
        macro = macro_average(results); macro["seed"] = seed
        per_seed_macro.append(macro)
        print(f"[seed {seed}] macro AP={macro['AP']:.4f} F1={macro['F1']:.4f} "
              f"PA_F1={macro['PA_F1']:.4f} EF1={macro['EF1']:.4f} FP1000={macro['FP1000']:.2f} "
              f"(AP over {macro['n_series_ap']}/{macro['n_series_total']} series, "
              f"{macro['n_series_no_pos']} series without positives, "
              f"{macro['n_test_points_total']} scored test points, "
              f"{n_no_valid} series had no validation windows for early stopping)")

    summary = summarise_across_seeds(per_seed_macro)
    _write_csv(out, per_series_rows, per_seed_macro, summary)
    print("\n=== Summary (mean +/- sample SD across seeds) ===")
    for key in METRIC_KEYS:
        print(f"{key:>6}: {summary[key+'_mean']:.4f} +/- {summary[key+'_std']:.4f}")
    print(f"\nWrote per_series.csv, per_seed_macro.csv, summary.csv to {out}/")
    return summary


def _write_csv(out, per_series_rows, per_seed_macro, summary):
    count_keys = ["n_series_ap", "n_series_prf", "n_series_no_pos",
                  "n_series_total", "n_test_points_total"]
    with open(os.path.join(out, "per_series.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["seed", "series"] + list(METRIC_KEYS) + ["n_test_points", "n_test_anom"])
        w.writerows(per_series_rows)
    with open(os.path.join(out, "per_seed_macro.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["seed"] + list(METRIC_KEYS) + count_keys)
        for m in per_seed_macro:
            w.writerow([m["seed"]] + [m[k] for k in METRIC_KEYS] + [m[k] for k in count_keys])
    with open(os.path.join(out, "summary.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["metric", "mean", "std"])
        for key in METRIC_KEYS:
            w.writerow([key, summary[key + "_mean"], summary[key + "_std"]])

In [9]:
NAB_ROOT = "NAB"
if not os.path.isdir(NAB_ROOT):
    print("[setup] cloning the Numenta Anomaly Benchmark ...")
    os.system("git clone --depth 1 https://github.com/numenta/NAB.git")

run_nab(NAB_ROOT, out="results_kanad_authors", seeds=(42, 43, 44), max_series=None)

[device] cpu   [params] {'window': 96, 'order': 2, 'batch_size': 1024, 'epochs': 100, 'lr': 0.01}   [trainable parameters] 274
[data] loaded 58 NAB series from NAB
  seed 42 [1/58] artificialNoAnomaly/art_daily_no_noise.csv: (no test anomaly) false alarm points per 1,000 = 10.42
  seed 42 [2/58] artificialNoAnomaly/art_daily_perfect_square_wave.csv: (no test anomaly) false alarm points per 1,000 = 10.42
  seed 42 [3/58] artificialNoAnomaly/art_daily_small_noise.csv: (no test anomaly) false alarm points per 1,000 = 8.93
  seed 42 [4/58] artificialNoAnomaly/art_flatline.csv: (no test anomaly) false alarm points per 1,000 = 0.00
  seed 42 [5/58] artificialNoAnomaly/art_noisy.csv: (no test anomaly) false alarm points per 1,000 = 13.89
  seed 42 [6/58] artificialWithAnomaly/art_daily_flatmiddle.csv: AP=0.200 F1=0.019 EF1=0.105
  seed 42 [7/58] artificialWithAnomaly/art_daily_jumpsdown.csv: AP=0.170 F1=0.005 EF1=0.111
  seed 42 [8/58] artificialWithAnomaly/art_daily_jumpsup.csv: AP=0.282 F1=

{'AP_mean': 0.17978382095042036,
 'AP_std': 0.0009024772930713855,
 'P_mean': 0.31520161283060993,
 'P_std': 0.015147219433480304,
 'R_mean': 0.047349517174103066,
 'R_std': 0.002137964575030419,
 'F1_mean': 0.07574578588622384,
 'F1_std': 0.0029830279419267376,
 'PA_F1_mean': 0.9260160801784547,
 'PA_F1_std': 0.0025477435820633267,
 'EP_mean': 0.09024379716636342,
 'EP_std': 0.008857014730304421,
 'ER_mean': 0.974468085106383,
 'ER_std': 0.004255319148936121,
 'EF1_mean': 0.14239559205699828,
 'EF1_std': 0.007033429321810339,
 'FP1000_mean': 13.125156491316547,
 'FP1000_std': 1.9413883394761884}

In [10]:
import inspect
print([l.strip() for l in inspect.getsource(evaluate_series).splitlines() if "pred = (" in l])

['pred = (np.asarray(test_scores) > threshold).astype(np.int64)']


In [11]:
s = [x for x in load_all_series("NAB") if "art_flatline" in x.name][0]
set_all_seeds(42)
sc = score_series(s.name, s.values, s.labels, AUTHOR_PARAMS, "cpu")
print("threshold      :", repr(percentile_threshold(sc.val_scores)))
print("val   distinct :", np.unique(sc.val_scores)[:5])
print("test  distinct :", np.unique(sc.test_scores)[:5])

[data] loaded 58 NAB series from NAB
threshold      : 0.13270506262779236
val   distinct : [0.13270506]
test  distinct : [0.13270506]


In [12]:
print((sc.test_scores > percentile_threshold(sc.val_scores)).sum())

0
